In [1]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

sys.path.append('../')
from layer_replace import *
from models import TinyCNN
from dataloader import cifar_dataset, get_imagenet_dataloaders
from linearize_rn18 import *

/storage/vsub851/anaconda3/envs/nas/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TinyCNN on CIFAR

In [2]:
model = TinyCNN()
model.load_state_dict(torch.load('../saved_models/tiny_cnn_cifar.pt'))
model

TinyCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 10, kernel_size=(1, 1), stride=(1, 1))
  )
  (gap): AdaptiveAvgPool2d(output_size=1)
)

In [3]:
target_type = nn.Conv2d
train_loader, _ = cifar_dataset(batch_size = 256)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
batch = next(iter(train_loader))
dummy_input = torch.randn(size = batch[0].shape)
input_shapes = extract_input_shapes(model, dummy_input = dummy_input, target_type = target_type)

In [5]:
replace_model_final = copy.deepcopy(model).to(device)
for idx in [1, 2, 3]:
    input_shape = input_shapes[idx - 1]
    replace_nth_module(replace_model_final, idx, input_shape, device, target_type, use_low_rank=True, rank=1024)

Replaced <class 'torch.nn.modules.conv.Conv2d'> #1 at Sequential(
  (0): LinearizedConv2d(
    (linear): LowRankLinear(
      (V): Linear(in_features=3072, out_features=1024, bias=False)
      (U): Linear(in_features=1024, out_features=32768, bias=True)
    )
  )
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 10, kernel_size=(1, 1), stride=(1, 1))
).0
Replaced <class 'torch.nn.modules.conv.Conv2d'> #2 at Sequential(
  (0): LinearizedConv2d(
    (linear): LowRankLinear(
      (V): Linear(in_features=3072, out_features=1024, bias=False)
      (U): Linear(in_features=1024, out_features=32768, bias=True)
    )
  )
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  

In [6]:
replace_model_final.load_state_dict(torch.load('../saved_models/orig_progressive_replace_cnn_conv_1_2_3.pt'))

<All keys matched successfully>

In [7]:
replace_model_final

TinyCNN(
  (features): Sequential(
    (0): LinearizedConv2d(
      (linear): LowRankLinear(
        (V): Linear(in_features=3072, out_features=1024, bias=False)
        (U): Linear(in_features=1024, out_features=32768, bias=True)
      )
    )
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): LinearizedConv2d(
      (linear): LowRankLinear(
        (V): Linear(in_features=8192, out_features=1024, bias=False)
        (U): Linear(in_features=1024, out_features=16384, bias=True)
      )
    )
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): LinearizedConv2d(
      (linear): LowRankLinear(
        (V): Linear(in_features=4096, out_features=1024, bias=False)
        (U): Linear(in_features=1024, out_features=640, bias=True)
      )
    )
  )
  (gap): AdaptiveAvgPool2d(output_size=1)
)

## ImageNet Loading (RN18 Theseus)

In [2]:
indices = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
orig = torchvision.models.resnet18(pretrained = True)

/storage/vsub851/anaconda3/envs/nas/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/storage/vsub851/anaconda3/envs/nas/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
target_type = nn.Conv2d
train_loader, _ = get_imagenet_dataloaders(batch_size = 256)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
orig = orig.to(device)

In [5]:
batch = next(iter(train_loader))
dummy_input = torch.randn(size = batch[0]['images'].permute(0, 3, 1, 2).shape)
input_shapes = extract_input_shapes(orig, dummy_input = dummy_input, target_type = target_type)

In [6]:
replace_model_final = copy.deepcopy(orig).to(device)
for idx in indices:
    input_shape = input_shapes[idx - 1]
    replace_nth_module(replace_model_final, idx, input_shape, device, target_type, use_low_rank=True, rank=1024)

Replaced <class 'torch.nn.modules.conv.Conv2d'> #3 at BasicBlock(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): LinearizedConv2d(
    (linear): LowRankLinear(
      (V): Linear(in_features=200704, out_features=1024, bias=False)
      (U): Linear(in_features=1024, out_features=200704, bias=True)
    )
  )
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
).conv2
Replaced <class 'torch.nn.modules.conv.Conv2d'> #4 at BasicBlock(
  (conv1): LinearizedConv2d(
    (linear): LowRankLinear(
      (V): Linear(in_features=200704, out_features=1024, bias=False)
      (U): Linear(in_features=1024, out_features=200704, bias=True)
    )
  )
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(64, 64, kernel_siz

In [7]:
replace_model_final.load_state_dict(torch.load('../saved_models/orig_progressive_replace_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20.pt'))

<All keys matched successfully>

In [8]:
indices = [1, 2]
for idx in indices:
    input_shape = input_shapes[idx - 1]
    replace_nth_module(replace_model_final, idx, input_shape, device, target_type, use_low_rank=True, rank=1024)

Replaced <class 'torch.nn.modules.conv.Conv2d'> #1 at ResNet(
  (conv1): LinearizedConv2d(
    (linear): LowRankLinear(
      (V): Linear(in_features=150528, out_features=1024, bias=False)
      (U): Linear(in_features=1024, out_features=802816, bias=True)
    )
  )
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): LinearizedConv2d(
        (linear): LowRankLinear(
          (V): Linear(in_features=200704, out_features=1024, bias=False)
          (U): Linear(in_features=1024, out_features=200704, bias=True)
        )
      )
      (bn2): BatchNorm2d(64, eps=1

In [10]:
torch.save(replace_model_final.state_dict(), '../saved_models/orig_progressive_replace_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20.pth')

In [17]:
replace_model_final.load_state_dict(torch.load('../saved_models/orig_progressive_replace_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20.pth'))

RuntimeError: Error(s) in loading state_dict for ResNet:
	size mismatch for conv1.linear.V.weight: copying a param with shape torch.Size([2048, 150528]) from checkpoint, the shape in current model is torch.Size([1024, 150528]).
	size mismatch for conv1.linear.U.weight: copying a param with shape torch.Size([802816, 2048]) from checkpoint, the shape in current model is torch.Size([802816, 1024]).
	size mismatch for layer1.0.conv1.linear.V.weight: copying a param with shape torch.Size([2048, 200704]) from checkpoint, the shape in current model is torch.Size([1024, 200704]).
	size mismatch for layer1.0.conv1.linear.U.weight: copying a param with shape torch.Size([200704, 2048]) from checkpoint, the shape in current model is torch.Size([200704, 1024]).